In [1]:
# CELL 0 – dùng chung cho NB01 → NB07
import os, json
from pathlib import Path

# notebooks/ nằm dưới thư mục gốc 1 cấp
PROJECT_ROOT = Path(os.getcwd()).parent
CONFIG_PATH = PROJECT_ROOT / "src" / "config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy {CONFIG_PATH}. Hãy chạy 00_prep_features.ipynb trước.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)

# Dùng Path để ghép đường dẫn cho tiện
RAW      = Path(cfg["RAW"])       # chỉ dùng ở NB01
FEATURES = Path(cfg["FEATURES"])
RESULTS  = Path(cfg["RESULTS"])
FIGURES  = Path(cfg["FIGURES"])

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW     :", RAW)
print("FEATURES:", FEATURES)
print("RESULTS :", RESULTS)
print("FIGURES :", FIGURES)


PROJECT_ROOT: D:\STAT3013.Q12_Group01
RAW     : D:\STAT3013.Q12_Group01\data\raw
FEATURES: D:\STAT3013.Q12_Group01\features
RESULTS : D:\STAT3013.Q12_Group01\results
FIGURES : D:\STAT3013.Q12_Group01\figures


In [2]:
import pandas as pd

tx_path = FEATURES / "tx_clean.parquet"
print("Đang kiểm tra file:", tx_path)

if tx_path.exists():
    tx = pd.read_parquet(tx_path)
    print(f" Đã tải dữ liệu tx: {tx.shape}")
else:
    raise FileNotFoundError(f"LỖI: Không tìm thấy file {tx_path}. Hãy chạy NB01 trước!")

# Đảm bảo các cột promo tồn tại và không bị NaN
for col in ["display", "mailer", "price_reduction"]:
    if col not in tx.columns:
        tx[col] = 0.0
    else:
        tx[col] = tx[col].fillna(0.0)

# --- Groupby tuần x department ---
weekly_dept = (
    tx.groupby(["week_no", "department"], as_index=False)
      .agg(
          qty=("quantity", "sum"),
          sales=("sales_value", "sum"),
          retail_disc=("retail_disc", "sum"),
          coupon_disc=("coupon_disc", "sum"),
          coupon_match_disc=("coupon_match_disc", "sum"),
          avg_price=("unit_price", "mean"),
          promo_display=("display", "mean"),
          promo_mailer=("mailer", "mean"),
          promo_price_red=("price_reduction", "mean"),
      )
)

weekly_dept["total_disc"] = (
    weekly_dept["retail_disc"].abs()
    + weekly_dept["coupon_disc"].abs()
    + weekly_dept["coupon_match_disc"].abs()
)
weekly_dept["gross_sales"] = weekly_dept["sales"] + weekly_dept["total_disc"]
weekly_dept["discount_rate"] = (
    weekly_dept["total_disc"] / weekly_dept["gross_sales"].replace(0, pd.NA)
)

weekly_dept = weekly_dept.rename(columns={"department": "group_id"})

print("Kích thước weekly_dept:", weekly_dept.shape)
weekly_dept.head()


Đang kiểm tra file: D:\STAT3013.Q12_Group01\features\tx_clean.parquet


 Đã tải dữ liệu tx: (1043543, 18)
Kích thước weekly_dept: (1083, 14)


,week_no,group_id,qty,sales,retail_disc,coupon_disc,coupon_match_disc,avg_price,promo_display,promo_mailer,promo_price_red,total_disc,gross_sales,discount_rate
0,1,AUTOMOTIVE,1,4.99,0.00,0.00,0.0,4.990000,0.0,0.0,0.0,0.00,4.99,0.0
1,1,COSMETICS,2,8.48,0.00,0.00,0.0,4.240000,0.0,0.0,0.0,0.00,8.48,0.0
2,1,DELI,43,171.02,-16.89,0.00,0.0,4.149342,0.0,0.0,0.0,16.89,187.91,0.089883
3,1,DRUG GM,297,822.85,-74.33,-0.59,0.0,2.737080,0.0,0.0,0.0,74.92,897.77,0.083451
4,1,FLORAL,6,63.94,-2.00,0.00,0.0,8.696667,0.0,0.0,0.0,2.00,65.94,0.030331


In [3]:
total_disc_w = (
    weekly_dept["retail_disc"].abs()
    + weekly_dept["coupon_disc"].abs()
    + weekly_dept["coupon_match_disc"].abs()
)

gross_sales_w = weekly_dept["sales"] + total_disc_w

weekly_dept["discount_rate"] = (total_disc_w / gross_sales_w.replace(0,1)).clip(0,1)
weekly_dept["weekofyear"] = (weekly_dept["week_no"] % 52).astype(int)

weekly_dept = weekly_dept.rename(columns={"department":"group_id"})
weekly_dept.head()


,week_no,group_id,qty,sales,retail_disc,coupon_disc,coupon_match_disc,avg_price,promo_display,promo_mailer,promo_price_red,total_disc,gross_sales,discount_rate,weekofyear
0,1,AUTOMOTIVE,1,4.99,0.00,0.00,0.0,4.990000,0.0,0.0,0.0,0.00,4.99,0.000000,1
1,1,COSMETICS,2,8.48,0.00,0.00,0.0,4.240000,0.0,0.0,0.0,0.00,8.48,0.000000,1
2,1,DELI,43,171.02,-16.89,0.00,0.0,4.149342,0.0,0.0,0.0,16.89,187.91,0.089883,1
3,1,DRUG GM,297,822.85,-74.33,-0.59,0.0,2.737080,0.0,0.0,0.0,74.92,897.77,0.083451,1
4,1,FLORAL,6,63.94,-2.00,0.00,0.0,8.696667,0.0,0.0,0.0,2.00,65.94,0.030331,1


In [4]:
FEATURES.mkdir(exist_ok=True)
out_path = FEATURES / "weekly_category.parquet"
weekly_dept.to_parquet(out_path, index=False)
print(" Đã lưu weekly_category.parquet vào:", out_path)



 Đã lưu weekly_category.parquet vào: D:\STAT3013.Q12_Group01\features\weekly_category.parquet
